In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:3]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-


In [3]:
openai = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key = openai_api_key
)

model = "openai/gpt-4o-mini"

In [4]:
system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [5]:
def chat(message, history):
    return "Pisang goreng"

In [7]:
# "messages" dalam type menentukan format bentuk history nantinya opsi bawaan dari gradio
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [8]:
def chat(message, history):
    return f"You said {message}, and the history is {history}, but I still say pisang goreng"

In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## How to write a slightly better chat callback!

In [22]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [39]:
system_message = """Kamu adalah asisten toko baju yang ramah. Doronglah pelanggan dengan halus \
untuk mencoba barang yang sedang diskon. Topi diskon 60%, dan kebanyakan barang lain diskon 50%. \
selalu panggil user dengan sebutan 'kak'
Selalu jawab dengan Bahasa Indonesia, gunakan bahasa yang santai, jangan gunakan bahasa baku.\n\n\
Berikut beberapa contoh cara menjawab:\n\
Contoh 1 — Pelanggan: 'Saya mau beli topi.'\n\
Jawaban: 'Wah, pilihan tepat! Kami punya banyak topi, dan beberapa di antaranya sedang diskon 60% kak.'\n\n\
Contoh 2 — Pelanggan: 'Ada jaket nggak?'\n\
Jawaban: 'ada kak! Semua jaket kami sedang diskon 50%. Mau saya carikan yang sesuai ukuranmu?'\n\n\
Contoh 3 — Pelanggan: 'Saya masih bingung mau beli apa.'\n\
Jawaban: 'Kalau gitu, saya sarankan lihat koleksi topi kita dulu kak — lagi diskon 60%, sayang kalau dilewatin!'\n\n\
Contoh 4 — Pelanggan: 'Ada sepatu yang diskon?'\n\
Jawaban: 'Untuk hari ini sepatu belum masuk diskon ya kak. Tapi kita punya topi yang lagi diskon 60%!'"""

In [19]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [20]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [23]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [40]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'buah' in message.lower():
        relevant_system_message += "The store does not buah if you are asked for buah, be sure to point out other items on sale."

    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]
    stream = openai.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content
        yield response

In [41]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.
